# Lab 0 · PyTorch warm-up: tensors, images, layers, training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-ORG/diffusion-workshop/blob/main/notebooks/00_pytorch_refresher.ipynb)

**Time:** about 20 minutes · **Goal:** make sure everyone can run a notebook on a GPU and knows the four PyTorch ideas the rest of the workshop leans on.

| You will | Why it matters later |
|---|---|
| treat an image as a tensor of numbers | a diffusion model adds and removes noise *per pixel* |
| add random noise to an image | this is the whole "forward" half of diffusion |
| watch layers change a tensor's shape | a U-Net is a chain of shape changes |
| write a training loop | every later lab reuses the same five lines |

Cells marked **TODO** contain `FIXME`. Replace each `FIXME` with code, then run the ✅ check cell underneath.

In [ ]:
# --- Workshop setup: run this cell first ------------------------------------
import os, sys

REPO_URL = "https://github.com/YOUR-ORG/diffusion-workshop.git"
if os.path.isdir("../diffusion_workshop"):            # running inside a local clone
    sys.path.insert(0, os.path.abspath(".."))
else:                                                 # running on Google Colab
    if not os.path.isdir("diffusion-workshop"):
        !git clone -q {REPO_URL} diffusion-workshop
    sys.path.insert(0, os.path.abspath("diffusion-workshop"))
    !pip -q install einops

import torch
import diffusion_workshop as dw
from diffusion_workshop import pick

device = dw.get_device()
dw.seed_everything(0)
print("device:", device, "| torch", torch.__version__)
if device.type != "cuda":
    print("No GPU found. On Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.")


## 1 · Tensors

A tensor is an n-dimensional array that can live on the GPU and remembers how it was computed (so PyTorch can compute gradients for us).

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print("shape :", x.shape)          # (rows, columns)
print("x * 2 :", x * 2)            # element-wise
print("mean  :", x.mean().item())
print("on    :", x.to(device).device)

`torch.randn` draws numbers from a standard normal distribution (mean 0, standard deviation 1). This **is** the "noise" of a diffusion model, so it is worth a look.

In [ ]:
import matplotlib.pyplot as plt

noise = torch.randn(10_000)
print(f"mean = {noise.mean():.3f}   std = {noise.std():.3f}")
plt.figure(figsize=(5, 2.5)); plt.hist(noise.numpy(), bins=60); plt.title("torch.randn"); plt.show()

## 2 · An image is a tensor

We use **FashionMNIST**: 60,000 small grayscale pictures of clothing, shrunk to 16×16 pixels so that training takes minutes instead of hours.

A batch of images has shape **(B, C, H, W)** = (batch, channels, height, width). Our pixel values are scaled to **[-1, 1]** so they are centred on zero, like the noise above.

In [ ]:
from diffusion_workshop.data import get_fashion_mnist, FASHION_LABELS
from diffusion_workshop.viz import show_images

IMG_SIZE = 16
dataset, loader = get_fashion_mnist(img_size=IMG_SIZE, batch_size=128)

images, labels = next(iter(loader))
print("batch shape :", tuple(images.shape))
print("value range :", images.min().item(), "to", images.max().item())
show_images(images[:16], titles=[FASHION_LABELS[i] for i in labels[:16]])

In [ ]:
# One image is just a grid of numbers. Here is the top-left 6x6 corner of the first one:
print(images[0, 0, :6, :6].round(decimals=1))

### TODO 1 · Add noise to an image

Make `noisy` = the images **plus** Gaussian noise scaled by `amount`.

Hint: `torch.randn_like(images)` gives noise with the same shape as `images`.

In [ ]:
amount = 0.5

noisy = images + amount * torch.randn_like(images)

show_images(torch.cat([images[:8], noisy[:8]]), ncols=8, suptitle="top: clean   bottom: noisy")

In [ ]:
# ✅ check
assert noisy.shape == images.shape, "noisy must have the same shape as images"
assert 0.3 < (noisy - images).std() < 0.7, "the noise should have a standard deviation of about `amount`"
print("✅ TODO 1 looks good")

## 3 · Layers change shapes

Three layers do most of the work in a U-Net:

| Layer | What it does to (B, C, H, W) |
|---|---|
| `nn.Conv2d(in, out, 3, padding=1)` | changes **C**, keeps H and W |
| `nn.MaxPool2d(2)` | keeps C, **halves** H and W |
| `nn.ConvTranspose2d(in, out, 2, stride=2)` | changes C, **doubles** H and W |

In [ ]:
import torch.nn as nn

x = images[:4]
print("input               ", tuple(x.shape))
h = nn.Conv2d(1, 8, kernel_size=3, padding=1)(x);           print("after Conv2d(1->8)  ", tuple(h.shape))
h = nn.MaxPool2d(2)(h);                                      print("after MaxPool2d(2)  ", tuple(h.shape))
h = nn.ConvTranspose2d(8, 4, kernel_size=2, stride=2)(h);    print("after ConvTranspose ", tuple(h.shape))

### TODO 2 · Hit a target shape

Build a small stack of layers that turns `(4, 1, 16, 16)` into `(4, 32, 4, 4)`.

Hint: you need to change the channels to 32 and halve the size twice.

In [ ]:
shrink = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.MaxPool2d(2),
    nn.MaxPool2d(2),
)
print(tuple(shrink(x).shape))

In [ ]:
# ✅ check
assert tuple(shrink(x).shape) == (4, 32, 4, 4), f"got {tuple(shrink(x).shape)}, wanted (4, 32, 4, 4)"
print("✅ TODO 2 looks good")

## 4 · The training loop

Every model in this workshop is trained with the same recipe:

1. run the model on a batch → **prediction**
2. compare with the target → **loss** (one number, lower is better)
3. `loss.backward()` → gradients (which way should each weight move?)
4. `optimizer.step()` → move the weights a little
5. `optimizer.zero_grad()` → forget the old gradients

We practise on a tiny clothing classifier.

### TODO 3 · Fill in the three optimizer lines

In [ ]:
import torch.nn.functional as F

classifier = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16x16 -> 8x8
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8x8  -> 4x4
    nn.Flatten(), nn.Linear(32 * 4 * 4, 10),
).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

losses = []
for step, (imgs, lbls) in enumerate(loader):
    imgs, lbls = imgs.to(device), lbls.to(device)
    loss = F.cross_entropy(classifier(imgs), lbls)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step >= pick(300, smoke=5):
        break

from diffusion_workshop.viz import plot_losses
plot_losses(losses, "Classifier loss")

In [ ]:
# ✅ check: the loss should fall well below where it started (about 2.3 = guessing at random)
if not dw.SMOKE:
    assert sum(losses[-20:]) / 20 < 1.0, "loss did not go down - check the three optimizer lines"
print(f"✅ TODO 3 looks good  (start {losses[0]:.2f} -> end {losses[-1]:.2f})")

## Recap

* Images are tensors shaped **(B, C, H, W)**, here with values in [-1, 1].
* **Noise** is `torch.randn_like(images)`: same shape, mean 0, std 1.
* Conv layers change channels; pooling halves the size; transposed convs double it.
* The training loop is always *predict → loss → backward → step → zero_grad*.

Next, **Lab 1**: we stack these layers into a U-Net and teach it to clean up noisy images.